# Task 1: Next-Word Prediction using MLP

## Importing the libraries

In [54]:

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from collections import Counter


torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

## Loading the Dataset

Let's start by loading and exploring our two datasets:

In [55]:
DATA_DIR = Path("data")
PG_ESSAYS_DIR = DATA_DIR / "paul_graham_essays"
LINUX_CODE_FILE = DATA_DIR / "python_code" / "linux_kernel_code.txt"

def load_paul_graham_essays():
    
    essays = []
    essay_files = []
    
    for file_path in PG_ESSAYS_DIR.glob("*.txt"):
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
            essays.append(content)
            essay_files.append(file_path.name)
    
    # Combine all essays
    combined_text = "\n\n".join(essays)
    
    print(f"Loaded {len(essays)} Paul Graham essays")
    print(f"Total characters: {len(combined_text):,}")
    print(f"Essay files: {essay_files}")
    
    return combined_text, essay_files

def load_linux_kernel_code():
    
    with open(LINUX_CODE_FILE, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    
    print(f"Loaded Linux kernel code")
    print(f"Total characters: {len(content):,}")
    
    return content

In [56]:
print("--- CATEGORY I: Paul Graham Essays ---")
pg_text, pg_files = load_paul_graham_essays()

print(f"\n--- CATEGORY II: Linux Kernel Code ---")
linux_text = load_linux_kernel_code()

--- CATEGORY I: Paul Graham Essays ---
Loaded 10 Paul Graham essays
Total characters: 192,049
Essay files: ['age of essay.txt', 'beating the averages.txt', 'do things that dont scale.txt', 'good and bad provrastination.txt', 'how to get startup ideas.txt', 'how to write usefully.txt', 'makers schedule.txt', 'the python paradox.txt', 'what you will wish you had known.txt', 'why nerds are unpopular.txt']

--- CATEGORY II: Linux Kernel Code ---
Loaded Linux kernel code
Total characters: 6,206,992


## 1.1 Pre Processing and Vocab

In [57]:
def preprocess_natural_language(text):

    text = re.sub(r'[^a-zA-Z0-9 \.]', '', text)
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    
    # Split into sentences using periods, then recombine
    sentences = [sent.strip() for sent in text.split('.') if sent.strip()]
    text = '. '.join(sentences) + '.'
    
    return text

def preprocess_code(text):
    text = text.lower()
    lines = text.split('\n')
    cleaned_lines = []
    
    for line in lines:
        #Keeping the indentation intact
        if line.strip():  
            cleaned_lines.append(' '.join(line.split()))
    
    text = '\n'.join(cleaned_lines)
    
    return text

In [58]:

print("--- Preprocessing Texts ---")


pg_clean = preprocess_natural_language(pg_text)
print("\nNatural Language")
print(f"Original length: {len(pg_text):,} chars")
print(f"Cleaned length: {len(pg_clean):,} chars")


linux_subset = linux_text[:500000] 
linux_clean = preprocess_code(linux_subset)
print("\nLinux Code")
print(f"Subset length: {len(linux_subset):,} chars")
print(f"Cleaned length: {len(linux_clean):,} chars")

print("\n---------------------------------------")
print(f"\n--- Cleaned Text Samples ---")
print(f"Paul Graham (first 200 chars): {pg_clean[:200]}...\n")
print(f"Linux code (first 200 chars): {linux_clean[:200]}...")

--- Preprocessing Texts ---

Natural Language
Original length: 192,049 chars
Cleaned length: 187,613 chars

Linux Code
Subset length: 500,000 chars
Cleaned length: 473,447 chars

---------------------------------------

--- Cleaned Text Samples ---
Paul Graham (first 200 chars): september 2004remember the essays you had to write in high school topic sentence introductory paragraph supporting paragraphs conclusion. the conclusion being say that ahab in moby dick was a christli...

Linux code (first 200 chars): /*
* linux/kernel/irq/autoprobe.c
*
* copyright (c) 1992, 1998-2004 linus torvalds, ingo molnar
*
* this file contains the interrupt probing code and driver apis.
*/
#include <linux/irq.h>
#include <l...


In [59]:
def build_vocabulary(text, min_freq=2):
    
    words = text.split()    
    word_counts = Counter(words)
    
    filtered_words = [word for word, count in word_counts.items() if count >= min_freq]
    
    # Create vocabulary mappings
    # Reserve index 0 for unknown words
    word_to_idx = {'<UNK>': 0}
    idx_to_word = {0: '<UNK>'}
    
    for i, word in enumerate(sorted(filtered_words), 1):
        word_to_idx[word] = i
        idx_to_word[i] = word
    
    return word_to_idx, idx_to_word, word_counts, words

In [60]:
print("Building Linux Code vocabulary")
lin_word_to_idx, lin_idx_to_word, lin_word_counts, lin_text = build_vocabulary(linux_clean, min_freq=2)
print(f"Vocabulary size: {len(lin_word_to_idx):,}")
print(lin_word_to_idx)
print('--------------')
print(lin_idx_to_word)
print('---------------')
print(lin_text)

Building Linux Code vocabulary
Vocabulary size: 4,678
{'<UNK>': 0, '!!match': 1, '!(signal->flags': 2, '!=': 3, '!config_debug_objects_rcu_head': 4, '!data->frozen': 5, '!defined(config_hotplug_cpu)': 6, '!error;': 7, '!kthread_should_stop())': 8, '!uid_eq(cred->uid,': 9, '"': 10, '""': 11, '""),': 12, '"#': 13, '"%s': 14, '"%s:': 15, '"%s\\n",': 16, '"((((((((a': 17, '"(a': 18, '"(e': 19, '");': 20, '",': 21, '".sys"': 22, '"0"))': 23, '"\\"");': 24, '"\\n");': 25, '"a': 26, '"bcdefgh"),': 27, '"bd"),': 28, '"bdfh"),': 29, '"before': 30, '"btt': 31, '"char"))': 32, '"duration': 33, '"e': 34, '"enable': 35, '"end': 36, '"failed': 37, '"holdoff': 38, '"illegal': 39, '"ip"))': 40, '"kernel': 41, '"number': 42, '"op_none");': 43, '"original"': 44, '"pm:': 45, '"power.h"': 46, '"resume"': 47, '"safe"': 48, '"safe",': 49, '"setting': 50, '"start': 51, '"syscall': 52, '"syscalls",': 53, '"test': 54, '"the': 55, '"time': 56, '"too': 57, '"trace.h"': 58, '"trace_output.h"': 59, '"use': 60, '#'

In [61]:
print("Building Paul Graham vocabulary")
pg_word_to_idx, pg_idx_to_word, pg_word_counts, pg_text = build_vocabulary(pg_clean, min_freq=2)
print(f"Vocabulary size: {len(pg_word_to_idx):,}")
print(pg_word_to_idx)
print('--------------')
print(pg_idx_to_word)
print('---------------')
print(pg_text)

Building Paul Graham vocabulary
Vocabulary size: 2,180
{'<UNK>': 0, '1': 1, '10': 2, '100': 3, '1876': 4, '1960s': 5, '1995': 6, '19th': 7, '1the': 8, '2': 9, '20': 10, '2001': 11, '2025': 12, '2the': 13, '3': 14, '4': 15, '5': 16, '50': 17, '6': 18, '7': 19, 'a': 20, 'abilities': 21, 'ability': 22, 'ability.': 23, 'able': 24, 'about': 25, 'about.': 26, 'absolute': 27, 'abstract': 28, 'abstractness': 29, 'academic': 30, 'accident.': 31, 'accounts': 32, 'achieve': 33, 'acknowledging': 34, 'acquired': 35, 'acquisition.': 36, 'across': 37, 'act': 38, 'actively': 39, 'actually': 40, 'added': 41, 'additional': 42, 'adjust': 43, 'admired': 44, 'admissions': 45, 'admit': 46, 'admitted': 47, 'adopters': 48, 'adult': 49, 'adults': 50, 'adults.': 51, 'advantage': 52, 'advantage.': 53, 'advice': 54, 'advise': 55, 'afford': 56, 'afraid': 57, 'after': 58, 'after.': 59, 'afternoon': 60, 'again': 61, 'against': 62, 'age': 63, 'age.': 64, 'ages': 65, 'aggressive': 66, 'ago': 67, 'agree': 68, 'agreed':

In [62]:
def report_vocabulary_stats(word_counts, vocab_name):
    print(f"{vocab_name} Vocabulary Statistics:")
    
    total_words = sum(word_counts.values())
    print(f"\nTotal words: {total_words:,}")
    
    # Most frequent words
    most_common = word_counts.most_common(10)
    print(f"\n10 Most Frequent Words:")
    for word, count in most_common:
        print(f"  {word}: {count}")
    
    # Least frequent words (with count > 1)
    least_common = [item for item in word_counts.most_common() if item[1] > 1][-10:]
    print(f"\n10 Least Frequent Words (count > 1):")
    for word, count in least_common:
        print(f"  {word}: {count}")

In [64]:
report_vocabulary_stats(pg_word_counts,"Paul Grahm Essays")
print("---------------------------------------------------------------------------------------")
report_vocabulary_stats(lin_word_counts,"Linux Kernel Code")

Paul Grahm Essays Vocabulary Statistics:

Total words: 33,978

10 Most Frequent Words:
  the: 1358
  to: 1192
  a: 875
  of: 735
  you: 732
  that: 609
  and: 547
  in: 529
  is: 484
  it: 425

10 Least Frequent Words (count > 1):
  apprentice: 2
  hierarchy.: 2
  court: 2
  veteran: 2
  noblesse: 2
  breeds: 2
  integrity: 2
  tactful: 2
  freaks.: 2
  rebellion.: 2
---------------------------------------------------------------------------------------
Linux Kernel Code Vocabulary Statistics:

Total words: 60,967

10 Most Frequent Words:
  *: 2635
  =: 2039
  {: 1438
  if: 1428
  }: 1335
  the: 1277
  */: 1202
  /*: 1079
  return: 847
  int: 788

10 Least Frequent Words (count > 1):
  mcount: 2
  seq_operations: 2
  *pos): 2
  inode->i_private);: 2
  ql,: 2
  g=%ld: 2
  rdp->cpu),: 2
  (int)(jiffies: 2
  0xffff));: 2
  *rnp;: 2
